# STRAT-412: Whole Foods Market Store Directory Scraper

This notebook collects all Whole Foods Market store locations and exports to CSV.

**Challenge:** The Whole Foods website is a JavaScript single-page app. Cloudscraper returns only the HTML shell, and Selenium has Chrome version issues in Colab.

**Solution:** We use the **OpenStreetMap Overpass API** — a free, public geographic database that has all Whole Foods locations with verified addresses. Returns structured JSON, no HTML scraping needed.

**Output CSV columns:** Store Name, Store Number, Store Complex, Address, City, State, Zip, Phone Number

## Imports & Setup

In [ ]:
!pip install requests --quiet

import requests
import json
import csv
import re
import time
from collections import Counter, defaultdict

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

OVERPASS_QUERY = """
[out:json][timeout:120];
area["ISO3166-1"="US"]->.usa;
(
  node["brand"="Whole Foods Market"]["shop"="supermarket"](area.usa);
  way["brand"="Whole Foods Market"]["shop"="supermarket"](area.usa);
  relation["brand"="Whole Foods Market"]["shop"="supermarket"](area.usa);
  node["name"="Whole Foods Market"]["shop"="supermarket"](area.usa);
  way["name"="Whole Foods Market"]["shop"="supermarket"](area.usa);
);
out body center;
"""

print("Setup complete.")
print("Data source: OpenStreetMap Overpass API (free, no key needed)")

## Discovery: Query the Overpass API

In [ ]:
print("Querying OpenStreetMap for all Whole Foods in the US...")
print("This may take 30-60 seconds.\n")

resp = requests.post(OVERPASS_URL, data={"data": OVERPASS_QUERY}, timeout=180)
resp.raise_for_status()
osm_data = resp.json()

elements = osm_data.get("elements", [])
print("Total OSM elements returned: " + str(len(elements)))

if elements:
    print("\nSample element tags:")
    print(json.dumps(elements[0].get("tags", {}), indent=2)[:600])

## Code Block #1: Scrape One Store

In [ ]:
def parse_osm_store(element):
    tags = element.get("tags", {})
    store_name = tags.get("name", tags.get("brand", "Whole Foods Market"))
    store_number = tags.get("ref", tags.get("brand:ref", ""))
    house_number = tags.get("addr:housenumber", "")
    street = tags.get("addr:street", "")
    address = (house_number + " " + street).strip() if (house_number or street) else tags.get("addr:full", "")
    city = tags.get("addr:city", "")
    state = tags.get("addr:state", "")
    zipcode = tags.get("addr:postcode", "")
    phone = tags.get("phone", tags.get("contact:phone", ""))
    store_complex = tags.get("located_in", tags.get("is_in", ""))
    zip_match = re.search(r"\d{5}", zipcode)
    if zip_match:
        zipcode = zip_match.group(0)
    state = state.strip().upper()[:2] if state else ""
    return {
        "Store Name": store_name,
        "Store Number": store_number,
        "Store Complex": store_complex,
        "Address": address,
        "City": city,
        "State": state,
        "Zip": zipcode,
        "Phone Number": phone.strip(),
    }

print("=" * 60)
print("SCRAPING ONE STORE")
print("=" * 60)
if elements:
    result = parse_osm_store(elements[0])
    for key, value in result.items():
        print("  " + key + ": " + value)

## Code Block #2: Test Small Loop (4 stores)

In [ ]:
print("=" * 60)
print("TESTING WITH SMALL LOOP (first 4 stores)")
print("=" * 60)
test_results = []
for i, el in enumerate(elements[:4]):
    print("\nStore " + str(i + 1) + ":")
    store_data = parse_osm_store(el)
    test_results.append(store_data)
    for key, value in store_data.items():
        print("  " + key + ": " + value)
print("\nParsed " + str(len(test_results)) + " test stores.")

## Code Block #3: Print States with Stores

In [ ]:
print("=" * 60)
print("STATES WITH WHOLE FOODS STORES")
print("=" * 60)
all_parsed = [parse_osm_store(el) for el in elements]
state_counts = Counter(s["State"] for s in all_parsed if s["State"])
for st, ct in sorted(state_counts.items()):
    print("  " + st + ": " + str(ct) + " stores")
print("\nTotal states: " + str(len(state_counts)))
missing = sum(1 for s in all_parsed if not s["State"])
if missing:
    print("Stores missing state: " + str(missing))

## Code Block #4: Print City-Level Entries

In [ ]:
print("=" * 60)
print("STORES BY CITY")
print("=" * 60)
sc_map = defaultdict(lambda: defaultdict(int))
for s in all_parsed:
    sc_map[s["State"] or "??"][s["City"] or "Unknown"] += 1
for st in sorted(sc_map.keys()):
    cities = sc_map[st]
    print("\n" + st + " (" + str(sum(cities.values())) + " stores):")
    for ct in sorted(cities.keys()):
        n = cities[ct]
        print("  " + ct + (" (" + str(n) + ")" if n > 1 else ""))

## Code Block #5: Print Full Store List

In [ ]:
print("=" * 60)
print("ALL STORES: " + str(len(all_parsed)))
print("=" * 60)
all_sorted = sorted(all_parsed, key=lambda x: (x["State"], x["City"], x["Store Name"]))
for i, s in enumerate(all_sorted, 1):
    print(str(i) + ". " + s["Store Name"] + " | #" + s["Store Number"]
          + " | " + s["Address"] + ", " + s["City"] + ", " + s["State"]
          + " " + s["Zip"] + " | " + s["Phone Number"])

## Code Block #6: Fill Missing Data via Reverse Geocoding
Uses OSM Nominatim to fill missing city/state/zip from lat/lon coordinates.

In [ ]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/reverse"
NOMINATIM_HEADERS = {"User-Agent": "STRAT412-ClassProject/1.0"}

STATE_ABBRS = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC",
}

def reverse_geocode(lat, lon):
    params = {"lat": lat, "lon": lon, "format": "json", "addressdetails": 1}
    r = requests.get(NOMINATIM_URL, params=params, headers=NOMINATIM_HEADERS, timeout=10)
    r.raise_for_status()
    return r.json()

print("=" * 60)
print("FILLING MISSING DATA VIA REVERSE GEOCODING")
print("=" * 60)

all_stores = []
geo_count = 0
for i, el in enumerate(elements):
    store = parse_osm_store(el)
    lat = el.get("lat") or el.get("center", {}).get("lat")
    lon = el.get("lon") or el.get("center", {}).get("lon")
    needs = not store["City"] or not store["State"] or not store["Zip"] or not store["Address"]
    if needs and lat and lon:
        try:
            time.sleep(1.1)
            geo = reverse_geocode(lat, lon)
            a = geo.get("address", {})
            if not store["Address"]:
                store["Address"] = (a.get("house_number", "") + " " + a.get("road", "")).strip()
            if not store["City"]:
                store["City"] = a.get("city", a.get("town", a.get("village", "")))
            if not store["State"]:
                store["State"] = STATE_ABBRS.get(a.get("state", ""), "")
            if not store["Zip"]:
                m = re.search(r"\d{5}", a.get("postcode", ""))
                if m:
                    store["Zip"] = m.group(0)
            geo_count += 1
        except Exception:
            pass
    all_stores.append(store)
    if (i + 1) % 50 == 0 or i == 0:
        print("  Processed " + str(i + 1) + "/" + str(len(elements)))

print("\nDone! " + str(len(all_stores)) + " stores. Geocoded " + str(geo_count) + " with missing data.")
print("Missing Address: " + str(sum(1 for s in all_stores if not s["Address"])))
print("Missing City:    " + str(sum(1 for s in all_stores if not s["City"])))
print("Missing State:   " + str(sum(1 for s in all_stores if not s["State"])))
print("Missing Zip:     " + str(sum(1 for s in all_stores if not s["Zip"])))
print("Missing Phone:   " + str(sum(1 for s in all_stores if not s["Phone Number"])))

## Code Block #7: Write to CSV and Export

In [ ]:
all_stores.sort(key=lambda x: (x["State"], x["City"], x["Store Name"]))

csv_filename = "whole_foods_stores.csv"
csv_columns = ["Store Name", "Store Number", "Store Complex", "Address", "City", "State", "Zip", "Phone Number"]

with open(csv_filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=csv_columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(all_stores)

print("CSV '" + csv_filename + "' written with " + str(len(all_stores)) + " rows.")
print("\nPreview (first 10):")
for s in all_stores[:10]:
    print("  " + s["Store Name"] + " | " + s["Address"] + ", " + s["City"] + ", " + s["State"] + " " + s["Zip"])

print("\nBy state:")
fc = Counter(s["State"] for s in all_stores if s["State"])
for st, ct in sorted(fc.items()):
    print("  " + st + ": " + str(ct))
print("Total: " + str(len(all_stores)) + " stores, " + str(len(fc)) + " states")

from google.colab import files
files.download(csv_filename)